# Univariate analysis: look at one variable at a time

_Data Wrangling_

**Before you relate two columns, know what each single column looks like — its shape, its spread, and its balance.**

Before you ask how two variables relate, you must know what each one is. Univariate
       analysis answers four questions about a single column's distribution:

       
         
- Shape &mdash; symmetric, or skewed (a long tail to one side)?
         
- Modality &mdash; one peak (unimodal), two peaks (bimodal), or more? Two peaks often means
         two hidden subgroups mixed together.
         
- Spread &mdash; tight around the center, or wide? (range, standard deviation, IQR
         (Inter-Quartile Range, the width of the middle 50% of the data)).
         
- Outliers &mdash; a few points far from the rest? Are they errors, or rare-but-real?
       

---

This notebook is a practice scaffold for the **Univariate analysis: look at one variable at a time** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — pandas + matplotlib + seaborn

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine

# --- Real bundled dataset: 178 wines, 13 numeric features + a 3-class target ---
d = load_wine(as_frame=True)
df = d.frame
print(df.shape)                      # -> (178, 14)

# =====================================================================
# 1) NUMERIC variable: look at ONE column four different ways.
#    'color_intensity' is right-skewed — a good column to study.
# =====================================================================
col = "color_intensity"
s = df[col]

# Summary stats first — but NEVER trust these alone; always plot too.
print(s.describe())
print("mean:", round(s.mean(), 2), " median:", round(s.median(), 2),
      " skew:", round(s.skew(), 2))   # mean > median, skew > 0  => right tail

fig, ax = plt.subplots(2, 2, figsize=(10, 7))

# (a) Histogram — vary the bin count; the real shape is stable across choices.
ax[0, 0].hist(s, bins=8, color="#79c0ff", edgecolor="white")
ax[0, 0].set_title("Histogram (8 bins) — right-skewed")

# (b) KDE — a smoothed density curve; good for spotting modality.
sns.kdeplot(s, ax=ax[0, 1], fill=True, color="#79c0ff")
ax[0, 1].set_title("KDE (smoothed distribution)")

# (c) Box plot — median, quartiles, and outliers beyond the whiskers.
ax[1, 0].boxplot(s, vert=False)
ax[1, 0].set_title("Box plot — spread + outliers")

# (d) ECDF — fraction of data at or below each value; read any percentile off it.
sns.ecdfplot(s, ax=ax[1, 1], color="#79c0ff")
ax[1, 1].set_title("ECDF — read percentiles directly")
plt.tight_layout(); plt.show()

# =====================================================================
# 2) CATEGORICAL summary: value_counts(normalize=True) + bar chart.
#    (We bucket a numeric column into bands to demo the categorical move;
#     for a real categorical column you would skip the cut.)
# =====================================================================
bands = pd.cut(df["alcohol"], bins=[0, 12, 13, 14, 99],
               labels=["<12", "12-13", "13-14", "14+"])
prop = bands.value_counts(normalize=True).sort_index()   # proportions, not raw counts
print(prop.round(3))
prop.plot.bar(color="#7ee787"); plt.title("alcohol band share"); plt.show()

# =====================================================================
# 3) THE TARGET: class-balance check (the plot that drives your metric).
#    For a CLASSIFICATION label, look at the share of each class.
# =====================================================================
target_share = df["target"].value_counts(normalize=True).sort_index()
print((target_share * 100).round(1))      # class_0 33%, class_1 40%, class_2 27%
df["target"].value_counts().sort_index().plot.bar(color="#ffa657")
plt.title("Target class balance — is any class rare?"); plt.xlabel("class"); plt.show()
# Read it: classes are fairly balanced here (~27-40%), so accuracy is a fair
# metric. If one class were <5%, accuracy would be a trap and you'd switch to
# recall / F1 / AUC. For a REGRESSION target, you'd histogram it and check skew.

## Visualize the data & results

_On sklearn's wine dataset, what does univariate analysis surface — the SHAPE of one numeric feature, and the CLASS BALANCE of the target?_

In [ ]:
import numpy as np
from sklearn.datasets import load_wine

d = load_wine(as_frame=True)
df = d.frame
print("shape:", df.shape)                 # -> (178, 14)

# --- LEFT chart: histogram (shape) of one numeric feature -------------
s = df["color_intensity"]
counts, edges = np.histogram(s, bins=8)
print("hist counts:", counts.tolist())    # -> [28, 45, 47, 25, 16, 10, 5, 2]
print("bin edges:", [round(e, 2) for e in edges])
print("mean:", round(s.mean(), 2),         # -> 5.06
      "median:", round(s.median(), 2),     # -> 4.69  (mean > median: right-skew)
      "skew:", round(s.skew(), 2))         # -> 0.87  (> 0: long right tail)

# --- RIGHT chart: class balance of the target -------------------------
counts = df["target"].value_counts().sort_index()
print("class counts:", counts.tolist())            # -> [59, 71, 48]
print("class share %:",
      (df["target"].value_counts(normalize=True).sort_index() * 100).round(1).tolist())
# -> [33.1, 39.9, 27.0]  -> fairly balanced, so accuracy is a fair metric here.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You plot a histogram of a price column with 5 bins and it looks like one smooth hill. A colleague plots the same data with 40 bins and clearly sees TWO separate peaks. Who is right, and what is the lesson?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recognize that bin count is a tunable knob, and too few bins can merge two peaks into one. — _5 wide bins average over the valley between the peaks, smearing a bimodal shape into one blob._
- Check whether the two peaks persist across several bin counts (and a KDE). — _Real structure is stable across sensible binnings; a binning artifact appears or vanishes as you change the count._
- If two peaks are stable, treat the variable as bimodal — likely two hidden subgroups mixed together. — _Two peaks often mean two populations (e.g. two product tiers), which matters for both analysis and modeling._

**Answer:** The colleague is closer to the truth: 5 bins were too coarse and hid a real bimodal
        structure. The lesson is never to trust a single bin count &mdash; vary it (and add a KDE). The shape
        that stays put across several sensible binnings is the real one. Two stable peaks usually mean two
        hidden subgroups, which changes how you model the variable.

</details>

**Problem 2.** A teammate trains a fraud classifier, reports 99.2% accuracy, and calls it done. You ask to see the target's class balance first. Why, and what likely went wrong?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Plot the target with value_counts(normalize=True) / a class-balance bar. — _Fraud is rare, so the label is almost certainly heavily imbalanced — say 99% legit, 1% fraud._
- Compare the 99.2% accuracy to the majority-class rate (~99%). — _A model that predicts "legit" for everyone already scores ~99% — beating it by 0.2% is essentially no skill at all._
- Switch to imbalance-robust metrics: precision, recall, F1, or AUC, and look at the confusion matrix. — _These measure whether the rare fraud class is actually being caught, which accuracy completely hides._

**Answer:** Because the target is almost surely severely imbalanced (fraud is rare), and on
        imbalanced data accuracy is a trap: always predicting the majority class already scores ~99%,
        so 99.2% may mean the model catches almost no fraud. Checking the class balance first reveals this;
        the fix is to judge the model with recall / precision / F1 / AUC and the confusion matrix, not
        accuracy.

</details>

**Problem 3.** For a numeric column you print .describe(): mean 52, median 31, max 4100. You report "average value about 52". What does the mean–median gap tell you, and how should you change the report and the modeling?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compare mean (52) to median (31): the mean is well above the median. — _Mean above median is the signature of a long right tail — the data is right-skewed._
- Note the huge max (4100) relative to the median. — _A few very large values are dragging the mean up; the mean is not a typical value here._
- Report the median, and consider a log transform before modeling. — _The median (31) is the honest typical value; a log transform pulls in the tail so models that assume symmetry behave._

**Answer:** Mean (52) far above median (31), with a max of 4100, says the variable is strongly
        right-skewed &mdash; a few large values drag the mean up, so 52 is not a typical value.
        Report the median (31) as the typical figure, and for modeling apply a log/power
        transform to pull in the tail. This is exactly why you plot and compare mean vs median instead of
        quoting the mean blindly.

</details>